# EDA — Indian Flight Delay

Análise exploratória que fundamenta os dois dashboards da aplicação Streamlit. O objetivo é entender qualidade, distribuição e relações entre variáveis e transformar os resultados em indicadores úteis para gestores de operações aéreas.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
data_path = Path('../dataset_voos/india_flight_delay_model_input.csv')
df = pd.read_csv(data_path, parse_dates=['Flight_Date'])
df.head()

## 1. Dimensões e qualidade dos dados

In [ ]:
print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}')
quality = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'ausentes': df.isna().sum(),
    'unicos': df.nunique()
})
print(f'Duplicatas: {df.duplicated().sum():,}')
quality

O conjunto contém 150 mil registros, não possui valores ausentes nem linhas duplicadas. `Flight_Number` é um identificador operacional e não é usado como medida quantitativa nos gráficos.

## 2. Estatísticas descritivas

In [ ]:
quantitative = [
    'Scheduled_Departure_Hour', 'Scheduled_Departure_Minute',
    'Temperature_C', 'Humidity_pct', 'Wind_Speed_kmh', 'Visibility_km',
    'Rainfall_mm', 'Cloud_Cover_pct', 'Origin_Congestion_Index',
    'Previous_Flight_Delay_Minutes', 'Turnaround_Risk_Index', 'Departure_Delay'
]
df[quantitative].describe().T

O atraso médio de partida é de aproximadamente 42,7 minutos. A mediana é 47 minutos, enquanto 13,4% dos voos não são marcados como atrasados. A variável de atraso apresenta concentração entre 39 e 54 minutos e uma cauda até 119 minutos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Departure_Delay'], bins=35, kde=True, ax=axes[0], color='#2F80ED')
axes[0].set_title('Distribuição do atraso na partida')
axes[0].set_xlabel('Atraso (minutos)')
sns.boxplot(data=df, x='Departure_Delay', ax=axes[1], color='#56CCF2')
axes[1].set_title('Dispersão e valores extremos')
axes[1].set_xlabel('Atraso (minutos)')
plt.tight_layout();

## 3. Correlações

In [ ]:
corr = df[quantitative].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True)
plt.title('Correlação de Pearson entre variáveis quantitativas')
plt.tight_layout();

In [ ]:
corr['Departure_Delay'].drop('Departure_Delay').sort_values(key=abs, ascending=False)

As correlações ajudam a priorizar hipóteses, mas não provam causalidade. O Dashboard 1 mantém essa análise interativa para permitir recortes de valores, companhias e clima.

## 4. Análises operacionais

In [ ]:
kpis = pd.Series({
    'Voos analisados': len(df),
    'Atraso médio (min)': df['Departure_Delay'].mean(),
    'Pontualidade até 15 min (%)': df['Departure_Delay'].le(15).mean() * 100,
    'Atraso severo a partir de 60 min (%)': df['Departure_Delay'].ge(60).mean() * 100,
    'Risco médio de turnaround': df['Turnaround_Risk_Index'].mean(),
})
kpis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
airline = df.groupby('Airline')['Departure_Delay'].mean().sort_values()
airline.plot.barh(ax=axes[0], color='#2F80ED', title='Atraso médio por companhia')
hour = df.groupby('Scheduled_Departure_Hour')['Departure_Delay'].mean()
hour.plot(ax=axes[1], marker='o', color='#F2994A', title='Atraso médio por hora')
weather = df.groupby('Weather')['Departure_Delay'].mean().sort_values(ascending=False)
weather.plot.bar(ax=axes[2], color='#EB5757', title='Atraso médio por clima')
for ax in axes: ax.set_ylabel('Minutos')
plt.tight_layout();

## 5. Conclusões e decisões para o app

- O **Dashboard 1** funciona como laboratório: variável selecionável, filtro de intervalo, histograma e mapa de calor.
- O **Dashboard 2** atende gestores de operações e monitora volume, atraso médio, pontualidade, atrasos severos e risco de turnaround.
- Os três visuais operacionais comparam companhia, hora de partida e clima.
- O dataset não contém dados pessoais de passageiros. Mesmo assim, o app evita publicar linhas individuais e apresenta apenas agregações.